# Organized Multi-Feature CamemBERT + FCNN Classifier

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CamembertTokenizer, CamembertModel
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import pickle
import os

In [14]:
X_kaggle = pd.read_pickle('data_kaggle.pkl')
y_train = pd.read_pickle('y_train.pkl')

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("\nLoading CamemBERT model and tokenizer...")

model_name = 'camembert-base'
tokenizer = CamembertTokenizer.from_pretrained(model_name)
camembert_model = CamembertModel.from_pretrained(model_name)
camembert_model.to(device)
camembert_model.eval()

for param in camembert_model.parameters():
    param.requires_grad = False

Using device: cuda

Loading CamemBERT model and tokenizer...


### 2.2 Define Embedding Extraction Function

In [4]:
def get_camembert_embeddings(texts, tokenizer, model, device, max_length=128, batch_size=32, desc="Extracting embeddings"):
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch_texts = texts[i:i+batch_size]
        
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(batch_embeddings)
    
    return np.vstack(embeddings)

In [5]:
text_features = ['full_text', 'source', 'user.location', 'user.description']
numerical_features = ['user.statuses_count', 'user.favourites_count', 'user.listed_count']

if os.path.exists('extracted_features.pkl'):
    print("Loading existing features from 'extracted_features.pkl'...")
    with open('extracted_features.pkl', 'rb') as f:
        features_dict = pickle.load(f)
    
    X_train_combined = features_dict['X_train_combined']
    X_kaggle_combined = features_dict['X_kaggle_combined']
    scaler = features_dict['scaler']
    feature_names = features_dict['feature_names']
    
    print(f"\nLoaded features:")
    print(f"  Train shape: {X_train_combined.shape}")
    print(f"  Kaggle shape: {X_kaggle_combined.shape}")
    print(f"  Feature breakdown: {feature_names}")
    print("\n✓ Features loaded successfully!")

else:
    # Prepare data (fill missing values)
    print("Preparing data...")
    X_train_prep = X_train.copy()
    X_kaggle_prep = X_kaggle.copy()
    
    # Text features - fill with empty string or 'unknown'
    for feature in text_features:
        if feature in ['full_text']:
            X_train_prep[feature] = X_train_prep[feature].fillna('')
            X_kaggle_prep[feature] = X_kaggle_prep[feature].fillna('')
        else:
            X_train_prep[feature] = X_train_prep[feature].fillna('unknown')
            X_kaggle_prep[feature] = X_kaggle_prep[feature].fillna('unknown')
    
    # Numerical features - fill with 0
    for feature in numerical_features:
        X_train_prep[feature] = X_train_prep[feature].fillna(0)
        X_kaggle_prep[feature] = X_kaggle_prep[feature].fillna(0)
    
    print("✓ Data preparation complete!\n")
    
    # Check for existing individual embedding files and load/extract as needed
    print("Checking for existing embedding files...")
    embeddings = {}
    missing_features = []
    
    # Mapping of feature names to their file names
    feature_file_mapping = {
        'full_text': 'embeddings_full_text',
        'source': 'embeddings_source',
        'user.location': 'embeddings_location',
        'user.description': 'embeddings_description'
    }
    
    for feature in text_features:
        file_base = feature_file_mapping[feature]
        train_file = f'{file_base}_train.pkl'
        kaggle_file = f'{file_base}_kaggle.pkl'
        
        if os.path.exists(train_file) and os.path.exists(kaggle_file):
            print(f"  ✓ Loading existing embeddings for '{feature}'...")
            with open(train_file, 'rb') as f:
                embeddings[f'train_{feature}'] = pickle.load(f)
            with open(kaggle_file, 'rb') as f:
                embeddings[f'kaggle_{feature}'] = pickle.load(f)
            print(f"    Shape: {embeddings[f'train_{feature}'].shape}")
        else:
            missing_features.append(feature)
    
    # Extract embeddings only for missing features
    if missing_features:
        print(f"\nExtracting CamemBERT embeddings for missing features: {missing_features}")
        for feature in missing_features:
            print(f"\n  Processing '{feature}'...")
            embeddings[f'train_{feature}'] = get_camembert_embeddings(
                X_train_prep[feature].tolist(),
                tokenizer,
                camembert_model,
                device,
                desc=f"Train {feature}"
            )
            embeddings[f'kaggle_{feature}'] = get_camembert_embeddings(
                X_kaggle_prep[feature].tolist(),
                tokenizer,
                camembert_model,
                device,
                desc=f"Kaggle {feature}"
            )
            
            # Save individual embeddings for future use
            file_base = feature_file_mapping[feature]
            with open(f'{file_base}_train.pkl', 'wb') as f:
                pickle.dump(embeddings[f'train_{feature}'], f)
            with open(f'{file_base}_kaggle.pkl', 'wb') as f:
                pickle.dump(embeddings[f'kaggle_{feature}'], f)
            
            print(f"  ✓ {feature}: {embeddings[f'train_{feature}'].shape} (saved)")
    else:
        print("\n✓ All embeddings loaded from existing files!")
    
    # Process numerical features
    print("\nProcessing numerical features...")
    
    # Check if scaler exists
    if os.path.exists('scaler_numerical.pkl'):
        print("  Loading existing scaler...")
        with open('scaler_numerical.pkl', 'rb') as f:
            scaler = pickle.load(f)
        train_numerical = scaler.transform(X_train_prep[numerical_features].values)
        kaggle_numerical = scaler.transform(X_kaggle_prep[numerical_features].values)
    else:
        print("  Creating new scaler...")
        scaler = StandardScaler()
        train_numerical = scaler.fit_transform(X_train_prep[numerical_features].values)
        kaggle_numerical = scaler.transform(X_kaggle_prep[numerical_features].values)
        with open('scaler_numerical.pkl', 'wb') as f:
            pickle.dump(scaler, f)
    
    print(f"  ✓ Scaled {len(numerical_features)} numerical features\n")
    
    # Combine all features
    print("Combining all features...")
    X_train_combined = np.hstack([
        embeddings['train_full_text'],          # 768 dimensions
        embeddings['train_source'],             # 768 dimensions
        embeddings['train_user.location'],      # 768 dimensions
        embeddings['train_user.description'],   # 768 dimensions
        train_numerical                          # 3 dimensions
    ])
    
    X_kaggle_combined = np.hstack([
        embeddings['kaggle_full_text'],
        embeddings['kaggle_source'],
        embeddings['kaggle_user.location'],
        embeddings['kaggle_user.description'],
        kaggle_numerical
    ])
    
    feature_names = (
        "full_text(768) + source(768) + user.location(768) + user.description(768) + "
        "user.statuses_count(1) + user.favourites_count(1) + user.listed_count(1) = "
        f"{X_train_combined.shape[1]} total"
    )
    
    print(f"  Train shape: {X_train_combined.shape}")
    print(f"  Kaggle shape: {X_kaggle_combined.shape}")
    print(f"  {feature_names}\n")
    
    # Save all features to single pickle file
    print("Saving unified features to 'extracted_features.pkl'...")
    features_dict = {
        'X_train_combined': X_train_combined,
        'X_kaggle_combined': X_kaggle_combined,
        'scaler': scaler,
        'feature_names': feature_names,
        'text_features': text_features,
        'numerical_features': numerical_features
    }
    
    with open('extracted_features.pkl', 'wb') as f:
        pickle.dump(features_dict, f)
    
    print("\n" + "="*70)
    print("✓ ALL FEATURES COMBINED AND SAVED SUCCESSFULLY!")
    print("  Future runs will load directly from 'extracted_features.pkl'")
    print("  You can now re-run training with different parameters by")
    print("  starting from the next cell.")
    print("="*70)

Loading existing features from 'extracted_features.pkl'...

Loaded features:
  Train shape: (154914, 3075)
  Kaggle shape: (103380, 3075)
  Feature breakdown: full_text(768) + source(768) + user.location(768) + user.description(768) + user.statuses_count(1) + user.favourites_count(1) + user.listed_count(1) = 3075 total

✓ Features loaded successfully!


## 4. Neural Network Architecture

In [6]:
# Custom Dataset for combined features
class CombinedFeatureDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels.values) if labels is not None else None
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        if self.labels is not None:
            return self.features[idx], self.labels[idx]
        return self.features[idx]

# Fully Connected Neural Network
class MultiFeatureFCNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout=0.3):
        super(MultiFeatureFCNN, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Build hidden layers
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        # Output layer (2 classes for binary classification)
        layers.append(nn.Linear(prev_dim, 2))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

print("Neural network classes defined!")

Neural network classes defined!


## 5. Training Configuration

In [7]:
# ============ TRAINING HYPERPARAMETERS ============
# Modify these to experiment with different configurations

BATCH_SIZE = 32
LEARNING_RATE = 0.001
NUM_EPOCHS = 15
HIDDEN_DIMS = [256, 128, 64]  # Network architecture
DROPOUT = 0.3
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5

# ==================================================

print("Training Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Hidden layers: {HIDDEN_DIMS}")
print(f"  Dropout: {DROPOUT}")

Training Configuration:
  Batch size: 32
  Learning rate: 0.001
  Epochs: 15
  Hidden layers: [256, 128, 64]
  Dropout: 0.3


## 6. Prepare Data Loaders

In [8]:
# Create datasets
print("Creating datasets...")
train_dataset = CombinedFeatureDataset(X_train_combined, y_train)
kaggle_dataset = CombinedFeatureDataset(X_kaggle_combined)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
kaggle_loader = DataLoader(kaggle_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Kaggle batches: {len(kaggle_loader)}")
print("Data loaders ready!")

Creating datasets...
Train batches: 4842
Kaggle batches: 3231
Data loaders ready!


## 7. Initialize Model and Training Components

In [9]:
# Initialize model
print("Initializing FCNN classifier...")
input_dim = X_train_combined.shape[1]
model = MultiFeatureFCNN(
    input_dim=input_dim,
    hidden_dims=HIDDEN_DIMS,
    dropout=DROPOUT
).to(device)

print(f"\nModel architecture: {input_dim} -> {' -> '.join(map(str, HIDDEN_DIMS))} -> 2")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE
)

print("\nTraining components initialized!")

Initializing FCNN classifier...

Model architecture: 3075 -> 256 -> 128 -> 64 -> 2
Total parameters: 829,634

Training components initialized!


## 8. Training and Prediction Functions

In [10]:
# Training function
def train_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for features, labels in tqdm(data_loader, desc="Training"):
        features = features.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(data_loader), correct / total

# Prediction function
def predict(model, data_loader, device):
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for features in tqdm(data_loader, desc="Predicting"):
            if isinstance(features, list):
                features = features[0]
            features = features.to(device)
            
            outputs = model(features)
            _, predicted = torch.max(outputs, 1)
            predictions.extend(predicted.cpu().numpy())
    
    return predictions

print("Training and prediction functions defined!")

Training and prediction functions defined!


## 9. Train the Model

In [11]:
# Train the model
print("\nTraining Multi-Feature FCNN model...")
print("="*70)
best_acc = 0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc * 100:.2f}%, LR: {current_lr:.6f}")
    
    # Learning rate scheduling
    scheduler.step(train_loss)
    
    if train_acc > best_acc:
        best_acc = train_acc
        # Save best model
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"✓ New best accuracy: {best_acc * 100:.2f}% - Model saved!")

print("\n" + "="*70)
print(f"Training complete! Best accuracy: {best_acc * 100:.2f}%")
print("="*70)


Training Multi-Feature FCNN model...

Epoch 1/15


Training: 100%|██████████| 4842/4842 [00:12<00:00, 388.67it/s]


Train Loss: 0.4662, Train Accuracy: 78.41%, LR: 0.001000
✓ New best accuracy: 78.41% - Model saved!

Epoch 2/15


Training: 100%|██████████| 4842/4842 [00:09<00:00, 512.68it/s]


Train Loss: 0.4398, Train Accuracy: 79.91%, LR: 0.001000
✓ New best accuracy: 79.91% - Model saved!

Epoch 3/15


Training: 100%|██████████| 4842/4842 [00:09<00:00, 525.44it/s]


... (21 outputs left out)

Training: 100%|██████████| 4842/4842 [00:09<00:00, 527.38it/s]


Train Loss: 0.4073, Train Accuracy: 81.66%, LR: 0.001000

Epoch 15/15


Training: 100%|██████████| 4842/4842 [00:09<00:00, 525.62it/s]

Train Loss: 0.4052, Train Accuracy: 81.84%, LR: 0.001000
✓ New best accuracy: 81.84% - Model saved!

Training complete! Best accuracy: 81.84%


## 10. Make Predictions and Save Results

In [15]:
# Load best model for predictions
print("Loading best model...")
model.load_state_dict(torch.load('best_model.pth'))

# Make predictions on Kaggle data
print("\nMaking predictions on Kaggle data...")
y_pred_test = predict(model, kaggle_loader, device)

# Prepare submission
output = pd.DataFrame({
    'ID': X_kaggle['challenge_id'],
    'Prediction': y_pred_test
})

# Save to CSV
output.to_csv('camembert_multifeature_fcnn_predictions.csv', index=False)
print("\n✓ Predictions saved to 'camembert_multifeature_fcnn_predictions.csv'")
print(f"\nPrediction distribution:")
print(pd.Series(y_pred_test).value_counts())
print(f"\nClass 0: {(pd.Series(y_pred_test) == 0).sum()} samples")
print(f"Class 1: {(pd.Series(y_pred_test) == 1).sum()} samples")

Loading best model...

Making predictions on Kaggle data...


Predicting: 100%|██████████| 3231/3231 [00:03<00:00, 939.14it/s] 



✓ Predictions saved to 'camembert_multifeature_fcnn_predictions.csv'

Prediction distribution:
0    59555
1    43825
Name: count, dtype: int64

Class 0: 59555 samples
Class 1: 43825 samples
